<a href="https://colab.research.google.com/github/segomezz/Practica-1-IA/blob/Segomezz-Ontolog%C3%ADas/Pr%C3%A1ctica1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2.3. Ontología y Razonamiento Semántico (RDFLib y OWL-RL)

In [2]:
!pip install rdflib
!pip install owlrl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 kB 2.0 MB/s eta 0:00:00


In [3]:
from rdflib import Graph, Namespace, RDF, RDFS, Literal, XSD, URIRef
from rdflib.namespace import FOAF, DC
from owlrl import DeductiveClosure, RDFS_Semantics

* Implementación de la Ontología del dominio utilizando RDF y RDFS.
  * El lenguaje de serialización de la ontología final desarrollada será en
Turtle.

  * Implementar al menos 10 clases (rdfs:Class) y 5 relaciones jerárquicas
(rdfs:subClassOf).

A continuación se realiza la definición de un diccionario llamada clases en el cual se establecen clase : subclases. Luego ese se recorre y se le da a cada clase y super clase su URI

In [4]:
# Namespaces
INV = Namespace("http://example.org/inversiones#")

# Crear el grafo
g = Graph()
g.bind("inv", INV)
g.bind("foaf", FOAF)
g.bind("dc", DC)

# Clases (10 clases + jerarquías)
clases = {
    "Inversion": None,
    "Activo": "Inversion",
    "Riesgo": "Inversion",
    "PerfilInversionista": "foaf:Person",
    "Recomendacion": None,
    "Accion": "Activo",
    "Bonos": "Activo",
    "Criptoactivo": "Activo",
    "InversionConservadora": "Inversion",
    "InversionAgresiva": "Inversion",
}

for clase, superclase in clases.items():
    clase_uri = INV[clase]
    g.add((clase_uri, RDF.type, RDFS.Class))
    if superclase:
        if ":" in superclase:
            prefix, name = superclase.split(":")
            super_uri = FOAF[name] if prefix == "foaf" else DC[name]
        else:
            super_uri = INV[superclase]
        g.add((clase_uri, RDFS.subClassOf, super_uri))



* Crear al menos 10 propiedades RDF originales con dominios
(rdfs:domain) y rangos (rdfs:range) correctamente definidos.
    * Incluir al menos un caso de jerarquía (rdfs:subPropertyOf)
    * Incluir como rangos valores tanto de tipo clase (URIs) y otros Tipo
Literales con sus respectivos tipados usando el XML-Schema.

  * Incluir el uso de clases o propiedades provenientes de vocabularios definidos y aceptados en la literatura (FOAF, Dublin Core, etc.).
  *Instanciar al menos 4 individuos por clase

  Se crea un Diccionario con las 10 propiedades y una subpropiedad. Acada propiedad se le asigna su Uri y de igual manera para el rango y el dominio

  Finalmente se realizan las 4 instancias

In [5]:
# Propiedades RDF (10 propiedades)
propiedades = {
    "tieneRiesgo": ("Inversion", "Riesgo"),
    "recomendadaPara": ("Recomendacion", "PerfilInversionista"),
    "tipoActivo": ("Inversion", "Activo"),
    "monto": ("Inversion", XSD.decimal),
    "rendimientoEsperado": ("Inversion", XSD.float),
    "edadInversionista": ("PerfilInversionista", XSD.integer),
    "nivelRiesgo": ("Riesgo", XSD.string),
    "recomienda": ("PerfilInversionista", "Recomendacion"),
    "descripcion": ("Inversion", XSD.string),
    "activoRelacionado": ("Recomendacion", "Activo"),
}
# Subproperty example
subproperties = {
    "activoEspecifico": "activoRelacionado",
}

for prop, (dom, ran) in propiedades.items():
    prop_uri = INV[prop]
    g.add((prop_uri, RDF.type, RDF.Property))
    g.add((prop_uri, RDFS.domain, INV[dom] if isinstance(dom, str) and ":" not in dom else FOAF[dom]))
    if isinstance(ran, URIRef):
        g.add((prop_uri, RDFS.range, ran))
    elif isinstance(ran, str) and ":" not in ran:
        g.add((prop_uri, RDFS.range, INV[ran]))
    else:
        g.add((prop_uri, RDFS.range, ran))

for child, parent in subproperties.items():
    g.add((INV[child], RDFS.subPropertyOf, INV[parent]))

# Instancias (4 por clase)
for i in range(1, 5):
    inv = INV[f"Inversion{i}"]
    g.add((inv, RDF.type, INV["InversionConservadora"]))
    g.add((inv, INV["descripcion"], Literal(f"Inversión conservadora {i}", datatype=XSD.string)))
    g.add((inv, INV["monto"], Literal(1000 * i, datatype=XSD.decimal)))
    g.add((inv, INV["rendimientoEsperado"], Literal(2.5 * i, datatype=XSD.float)))
    g.add((inv, INV["tipoActivo"], INV["Bonos"]))

    perfil = INV[f"Inversionista{i}"]
    g.add((perfil, RDF.type, INV["PerfilInversionista"]))
    g.add((perfil, FOAF.name, Literal(f"Inversionista {i}")))
    g.add((perfil, INV["edadInversionista"], Literal(30 + i, datatype=XSD.integer)))

    riesgo = INV[f"Riesgo{i}"]
    g.add((riesgo, RDF.type, INV["Riesgo"]))
    g.add((riesgo, INV["nivelRiesgo"], Literal("Bajo", datatype=XSD.string)))
    g.add((inv, INV["tieneRiesgo"], riesgo))

    recomendacion = INV[f"Recomendacion{i}"]
    g.add((recomendacion, RDF.type, INV["Recomendacion"]))
    g.add((recomendacion, INV["recomendadaPara"], perfil))
    g.add((recomendacion, INV["activoEspecifico"], INV["Bonos"]))


  * Aplicar razonamiento sobre la ontología RDFS anteriormente desarrollada usando DeductiveClosure(RDFS Semantics) de owlrl.
    * Completar de ser necesario la antología desarrollada para realizar lo siguiente:
    ** Documentar al menos dos casos de generación de nuevos hechos asociados a una jerarquía de clases (ver caso 1 de las diapositivas)

    * * Documentar la asociación de un caso de miembro de una clase desde el dominio o rango de sus propiedades (ver caso 2 de las diapositivas)
    ** Documentar un caso de nuevos hechos desde la relación
subproperty (ver caso 3 de las diapositivas)
    ** Todos los tres requerimientos anteriores, deben ser generados al usar el razonador OWL-RL usando DeductiveClosure(RDFS Semantics) sobre la ontología final de su dominio.

    * Comparar el grafo antes y después del razonamiento para evidenciar nuevas afirmaciones inferidas automáticamente.
    * Adicionalmente a lo anteriormente requerido se debe evidenciar al menos 4 nuevos hechos inferidos generados como resultado del razonamiento.

In [6]:
# Grafo original
antes_razonamiento = set(g.serialize(format="nt").splitlines())
print(g.serialize(format="turtle"))

@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix inv: <http://example.org/inversiones#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

inv:Accion a rdfs:Class ;
    rdfs:subClassOf inv:Activo .

inv:Activo a rdfs:Class ;
    rdfs:subClassOf inv:Inversion .

inv:Bonos a rdfs:Class ;
    rdfs:subClassOf inv:Activo .

inv:Criptoactivo a rdfs:Class ;
    rdfs:subClassOf inv:Activo .

inv:Inversion a rdfs:Class .

inv:InversionAgresiva a rdfs:Class ;
    rdfs:subClassOf inv:Inversion .

inv:InversionConservadora a rdfs:Class ;
    rdfs:subClassOf inv:Inversion .

inv:PerfilInversionista a rdfs:Class ;
    rdfs:subClassOf foaf:Person .

inv:Recomendacion a rdfs:Class .

inv:Riesgo a rdfs:Class ;
    rdfs:subClassOf inv:Inversion .

inv:Inversion1 a inv:InversionConservadora ;
    inv:descripcion "Inversión conservadora 1"^^xsd:string ;
    inv:monto 1000.0 ;
    inv

In [7]:
# Razonamiento
DeductiveClosure(RDFS_Semantics).expand(g)

In [13]:

print(g.serialize(format="turtle"))




@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix inv: <http://example.org/inversiones#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

inv:Accion a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Accion,
        inv:Activo,
        inv:Inversion,
        rdfs:Resource .

inv:Activo a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Activo,
        inv:Inversion,
        rdfs:Resource .

inv:Bonos a inv:Activo,
        inv:Inversion,
        rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Activo,
        inv:Bonos,
        inv:Inversion,
        rdfs:Resource .

inv:Criptoactivo a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Activo,
        inv:Criptoactivo,
        inv:Inversion,
        rdfs:Resource .

inv:Inversion a rdfs:Class,
        rdfs:Resource ;
    rdfs:subClassOf inv:Inversion,
        rdfs:Resource